# PennyLane MPS convergence evidence

Compare a long-range 10-wire QNode with default.qubit and inspect MettleQ Dmax convergence metadata.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [ ]:
rng = np.random.default_rng(52)
angles = rng.uniform(-0.8, 0.8, size=(3, 10))
pairs = [[tuple(map(int, pair)) for pair in rng.permutation(10).reshape(5, 2)] for _ in range(3)]

def make_qnode(device):
    @qml.qnode(device)
    def circuit():
        for layer in range(3):
            for wire in range(10):
                qml.RY(angles[layer, wire], wires=wire)
            for first, second in pairs[layer]:
                qml.IsingZZ(0.43, wires=[first, second])
        return qml.state(), qml.expval(qml.Z(0))
    return circuit

reference_qnode = make_qnode(qml.device("default.qubit", wires=10))
reference, reference_ms, _ = benchmark(reference_qnode)
mettleq_device = MettleQDevice(
    wires=10,
    method="matrix_product_state",
    device="cpu",
    mps_max_bond_dimension=32,
    mps_truncation_threshold=1e-12,
    mps_convergence_bond_dimensions=(8, 16, 32),
    mps_convergence_atol=5e-4,
)
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(mettleq_qnode)
state_error = phase_aligned_statevector_error(reference[0], candidate[0])
expectation_error = abs(float(reference[1]) - float(candidate[1]))
convergence = mettleq_device.last_mps_convergence_report
accuracy = mettleq_device.last_mps_accuracy_report
convergence_summary = {
    "converged": convergence["converged"],
    "atol": convergence["atol"],
    "comparisons": convergence["comparisons"],
    "runs": [
        {
            "dmax": run["dmax"],
            "accuracy_classification": run["accuracy"]["classification"],
            "maximum_bond_dimension_reached": run["diagnostics"]["maximum_bond_dimension_reached"],
        }
        for run in convergence["runs"]
    ],
}
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/12_mps_convergence.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="MPS state atol=8e-5 and Dmax convergence atol=5e-4",
    passed=state_error <= 8e-5 and expectation_error <= 5e-5 and convergence["converged"] and accuracy["passed"],
    exact_match=bool(np.array_equal(reference[0], candidate[0])),
    selected_method=method,
    selected_device=device,
    metrics={"max_amplitude_error": state_error, "expectation_error": expectation_error, "accuracy": accuracy, "convergence": convergence_summary},
)